# ✈️ Airline Delay Causes — Data Cleaning
**Dataset:** [Kaggle - Airline Delay Causes](https://www.kaggle.com/datasets/giovamata/airlinedelaycauses)

> تأكد إن ملف `DelayedFlights.csv` موجود في نفس الفولدر

---


## 📦 1. استيراد المكتبات

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
print("✅ Libraries loaded")

## 📂 2. تحميل الداتا

In [ ]:
df = pd.read_csv("DelayedFlights.csv")

print(f"الصفوف: {df.shape[0]:,}  |  الأعمدة: {df.shape[1]}")
df.head()

## 🔍 3. معلومات الداتا الأولية

In [ ]:
df.info()

In [ ]:
df.describe().T.style.background_gradient(cmap="Blues")

## ❓ 4. القيم المفقودة (Missing Values)

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
}).query("`Missing Count` > 0").sort_values("Missing %", ascending=False)

missing_report.style.background_gradient(cmap="Reds", subset=["Missing %"])

In [ ]:
plt.figure(figsize=(14, 4))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap="viridis")
plt.title("Missing Values Heatmap", fontsize=13)
plt.tight_layout()
plt.show()

## 🛠️ 5. معالجة القيم المفقودة

| العمود | الاستراتيجية | السبب |
|--------|-------------|-------|
| CarrierDelay / WeatherDelay / NASDelay / SecurityDelay / LateAircraftDelay | تعبئة بـ `0` | Missing = مفيش تأخير من السبب ده |
| CancellationCode | تعبئة بـ `"Not Cancelled"` | الرحلة مش متلغية |
| TailNum | تعبئة بـ `"UNKNOWN"` | معرفش كود الطيارة |
| AirTime / ActualElapsedTime | تعبئة بالـ **median** | قيم رقمية، median أكتر مقاومة للـ outliers |


In [ ]:
delay_cause_cols = [
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay"
]

# ── Delay causes → 0
for col in delay_cause_cols:
    if col in df.columns:
        before = df[col].isnull().sum()
        df[col] = df[col].fillna(0)
        print(f"  ✅ {col}: filled {before:,} NaN → 0")

# ── CancellationCode
if "CancellationCode" in df.columns:
    before = df["CancellationCode"].isnull().sum()
    df["CancellationCode"] = df["CancellationCode"].fillna("Not Cancelled")
    print(f"  ✅ CancellationCode: filled {before:,} NaN → 'Not Cancelled'")

# ── TailNum
if "TailNum" in df.columns:
    before = df["TailNum"].isnull().sum()
    df["TailNum"] = df["TailNum"].fillna("UNKNOWN")
    print(f"  ✅ TailNum: filled {before:,} NaN → 'UNKNOWN'")

# ── AirTime & ActualElapsedTime → median
for col in ["AirTime", "ActualElapsedTime"]:
    if col in df.columns:
        med = df[col].median()
        before = df[col].isnull().sum()
        df[col] = df[col].fillna(med)
        print(f"  ✅ {col}: filled {before:,} NaN → median ({med:.1f})")

In [ ]:
# تحقق: كام قيمة مفقودة فضلت؟
remaining = df.isnull().sum().sum()
print(f"إجمالي القيم المفقودة المتبقية: {remaining:,}")

## 🔁 6. الصفوف المكررة (Duplicates)

In [ ]:
dups = df.duplicated().sum()
print(f"عدد الصفوف المكررة: {dups:,}")

if dups > 0:
    df = df.drop_duplicates()
    print(f"  ✅ تم حذف {dups:,} صف مكرر")
else:
    print("  ✅ مفيش صفوف مكررة")

## 📐 7. تصحيح أنواع البيانات (Data Types)

In [ ]:
int_cols = ["Year", "Month", "DayofMonth", "DayOfWeek", "FlightNum", "Distance"]
float_cols = ["DepDelay", "ArrDelay", "AirTime", "ActualElapsedTime", "CRSElapsedTime"] + delay_cause_cols

for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

for col in float_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(" Data types corrected")
df.dtypes

## 📈 8. كشف الـ Outliers (IQR Method)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for i, col in enumerate(["DepDelay", "ArrDelay"]):
    if col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 3 * IQR
        upper = Q3 + 3 * IQR
        outliers = df[(df[col] < lower) | (df[col] > upper)].shape[0]

        axes[i].boxplot(df[col].dropna(), vert=False, patch_artist=True,
                        boxprops=dict(facecolor="#4C72B0", alpha=0.6))
        axes[i].set_title(f"{col}\nOutliers: {outliers:,} ({outliers/len(df)*100:.2f}%)")
        axes[i].set_xlabel("دقائق")
        print(f"{col}: lower={lower:.0f}, upper={upper:.0f}, outliers={outliers:,}")

plt.suptitle("Outlier Detection — Delay Columns", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## ⚙️ 9. Feature Engineering — أعمدة جديدة مفيدة

In [ ]:
# هل الرحلة اتأخرت؟ (FAA standard: >15 min)
if "ArrDelay" in df.columns:
    df["IsDelayed"] = (df["ArrDelay"] > 15).astype(int)

# فئة التأخير
def classify_delay(delay):
    if pd.isna(delay) or delay <= 0:
        return "On Time / Early"
    elif delay <= 15:
        return "Minor (1-15 min)"
    elif delay <= 60:
        return "Moderate (16-60 min)"
    else:
        return "Severe (>60 min)"

df["DelayCategory"] = df["ArrDelay"].apply(classify_delay)

# الفصل
def get_season(month):
    if pd.isna(month): return "Unknown"
    m = int(month)
    return "Winter" if m in [12,1,2] else "Spring" if m in [3,4,5] else "Summer" if m in [6,7,8] else "Fall"

df["Season"] = df["Month"].apply(get_season)

print("✅ New columns: IsDelayed, DelayCategory, Season")
df[["ArrDelay", "IsDelayed", "DelayCategory", "Season"]].head(10)

## 📊 10. رسومات بيانية بعد الكلينينج

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("✈️ Airline Delays — Overview After Cleaning", fontsize=15, fontweight="bold", y=1.01)

# 1: توزيع ArrDelay
data_plot = df["ArrDelay"].clip(-60, 300)
axes[0,0].hist(data_plot, bins=60, color="#4C72B0", edgecolor="white", alpha=0.85)
axes[0,0].set_title("Distribution of Arrival Delay")
axes[0,0].set_xlabel("Minutes"); axes[0,0].set_ylabel("Flights")

# 2: فئات التأخير
cat_order = ["On Time / Early","Minor (1-15 min)","Moderate (16-60 min)","Severe (>60 min)"]
cat_counts = df["DelayCategory"].value_counts().reindex(cat_order)
colors = ["#2ecc71","#f1c40f","#e67e22","#e74c3c"]
axes[0,1].bar(cat_counts.index, cat_counts.values, color=colors)
axes[0,1].set_title("Delay Categories")
axes[0,1].tick_params(axis="x", rotation=20)

# 3: متوسط التأخير حسب الشهر
monthly = df.groupby("Month")["ArrDelay"].mean()
axes[1,0].plot(monthly.index, monthly.values, marker="o", color="#E74C3C", linewidth=2.5)
axes[1,0].set_title("Avg Arrival Delay by Month")
axes[1,0].set_xlabel("Month"); axes[1,0].set_ylabel("Avg Minutes")
axes[1,0].grid(alpha=0.3)

# 4: أسباب التأخير
cause_means = {c.replace("Delay",""): df[c].mean() for c in delay_cause_cols if c in df.columns}
axes[1,1].barh(list(cause_means.keys()), list(cause_means.values()), color="#9B59B6")
axes[1,1].set_title("Avg Delay by Cause (minutes)")
axes[1,1].set_xlabel("Minutes")

plt.tight_layout()
plt.savefig("airline_cleaning_report.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Saved: airline_cleaning_report.png")

## ✅ 11. التقرير النهائي

In [ ]:
print("=" * 55)
print("📋 FINAL CLEANING REPORT")
print("=" * 55)
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")
print(f"Missing : {df.isnull().sum().sum():,} values remaining")
print(f"Delayed flights (>15 min): {df['IsDelayed'].sum():,} ({df['IsDelayed'].mean()*100:.1f}%)")
print("=" * 55)
df.dtypes

## 💾 12. حفظ الداتا النظيفة

In [ ]:
df.to_csv("DelayedFlights_cleaned.csv", index=False)
print("✅ Saved: DelayedFlights_cleaned.csv")